In [2]:
# Load data
import pandas as pd

df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")  # ganti sesuai nama file kamu
print(df.shape)
print(df.columns.tolist())
df.head()

(2111, 17)
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']


,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [3]:
# =========================
# 1. IMPORT
# =========================
import numpy as np
import pandas as pd
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)


# =========================
# 2. TARGET DAN FITUR
# =========================
target_col = "NObeyesdad"

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].astype(str)

# Encode target class menjadi angka 0..K-1
y_le = LabelEncoder()
y = y_le.fit_transform(y_raw)
n_classes = len(np.unique(y))

print("Classes:", list(y_le.classes_))
print("Jumlah kelas:", n_classes)


# =========================
# 3. IDENTIFIKASI KOLOM
# =========================
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Numerical columns :", num_cols)
print("Categorical columns:", cat_cols)


# =========================
# 4. SPLIT DATA
# train : val : test = 68 : 12 : 20
# =========================
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.15,   # 15% dari train_full
    random_state=42,
    stratify=y_train_full
)

print("\nUkuran data:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


# =========================
# 5. PREPROCESSING
# - Numeric -> StandardScaler
# - Categorical -> OneHotEncoder
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

X_train_nn = preprocess.fit_transform(X_train)
X_val_nn   = preprocess.transform(X_val)
X_test_nn  = preprocess.transform(X_test)

# Kalau hasil sparse, ubah ke dense untuk Keras
X_train_nn = X_train_nn.toarray() if hasattr(X_train_nn, "toarray") else X_train_nn
X_val_nn   = X_val_nn.toarray() if hasattr(X_val_nn, "toarray") else X_val_nn
X_test_nn  = X_test_nn.toarray() if hasattr(X_test_nn, "toarray") else X_test_nn

n_features = X_train_nn.shape[1]
print("\nJumlah fitur setelah preprocessing:", n_features)


# =========================
# 6. BANGUN MODEL MLP
# versi moderat: 256 -> 128 -> 64
# =========================
mlp = keras.Sequential([
    layers.Input(shape=(n_features,)),

    layers.Dense(256),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.30),

    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.20),

    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.10),

    layers.Dense(n_classes, activation="softmax")
])

mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mlp.summary()


# =========================
# 7. CALLBACKS
# =========================
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=15,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-5
    )
]


# =========================
# 8. TRAINING
# =========================
start_train = time.time()

history = mlp.fit(
    X_train_nn, y_train,
    validation_data=(X_val_nn, y_val),
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

rt_train_nn = time.time() - start_train


# =========================
# 9. PREDIKSI
# =========================
start_pred = time.time()

y_pred_prob = mlp.predict(X_test_nn, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

rt_pred_nn = time.time() - start_pred


# =========================
# 10. EVALUASI
# =========================
acc_nn = accuracy_score(y_test, y_pred)

print("\n=== HASIL MLP ===")
print(f"Accuracy       : {acc_nn:.4f} ({acc_nn*100:.2f}%)")
print(f"Runtime train  : {rt_train_nn:.3f} s")
print(f"Runtime predict: {rt_pred_nn:.3f} s")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=y_le.classes_))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

Classes: ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']
Jumlah kelas: 7
Numerical columns : ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categorical columns: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

Ukuran data:
X_train: (1434, 16)
X_val  : (254, 16)
X_test : (423, 16)

Jumlah fitur setelah preprocessing: 31


C:\Users\asus\AppData\Local\Temp\ipykernel_21116\2759899051.py:43: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,591 (201.53 KB)

 Trainable params: 50,695 (198.03 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.3340 - loss: 1.7686 - val_accuracy: 0.5866 - val_loss: 1.7021 - learning_rate: 0.0010
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5725 - loss: 1.1582 - val_accuracy: 0.7008 - val_loss: 1.4456 - learning_rate: 0.0010
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6890 - loss: 0.9209 - val_accuracy: 0.7480 - val_loss: 1.2422 - learning_rate: 0.0010
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7448 - loss: 0.7944 - val_accuracy: 0.7717 - val_loss: 1.0707 - learning_rate: 0.0010
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7866 - loss: 0.6902 - val_accuracy: 0.7835 - val_loss: 0.9264 - learning_rate: 0.0010
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7796 - loss: 0.6281 - val_accuracy: 0.8189 - val_loss: 0.8009 - learning_rate: 0.0010
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8173 - loss: 0.5653 - val_ac